### Volume e Percentual Geral de Atrasos
**Pergunta de Negócio:** Qual é o volume e o percentual de pedidos entregues com atraso na operação?

**Métricas Calculadas:**
- Quantidade total de pedidos por status de entrega (`qtd_pedidos`).
- Percentual representativo em relação ao total da base (`percentual`).


In [0]:
%sql

SELECT 
  late_delivery,
  COUNT(*) AS qtd_pedidos,
  ROUND (
    COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 
    2
  ) as percentual
FROM workspace.default.entregas_tratadas
GROUP BY late_delivery
ORDER BY late_delivery;

**Insight / Resultado:**
Dos 50.000 pedidos analisados, **28.162 (56,32%) foram entregues com atraso**, enquanto 21.838 (43,68%) foram entregues dentro do prazo. A taxa de atraso supera a metade do volume total da operação, indicando um gargalo sistêmico na cadeia logística.

###  Severidade e Duração dos Atrasos
**Pergunta de Negócio:** Quantos dias de atraso os pedidos em atraso apresentam, em média, e qual a severidade máxima observada?

**Métricas Calculadas:**
- Média de dias de atraso (`media_dias_atraso`).
- Mediana para mitigar distorções de valores extremos (`mediana_dias_atraso`).
- Atraso máximo registrado na base (`maior_atraso_dias`).

In [0]:
%sql
SELECT 
    ROUND(AVG (delivery_delay_days),2) AS media_dias_atraso,
    MEDIAN(delivery_delay_days) AS mediana_dias_atraso,
    MAX(delivery_delay_days) AS maior_atraso_dias
FROM workspace.default.entregas_tratadas
WHERE late_delivery = 'Yes';

**Insight / Resultado:**
Entre os pedidos atrasados, o tempo médio de atraso é de **2,27 dias**, com uma mediana de **2 dias**. O maior atraso individual registrado foi de **13 dias**. A proximidade entre a média e a mediana indica uma distribuição consistente nos prazos de atraso, sem grandes disrupções isoladas.

### Distribuição Temporal dos Atrasos
**Pergunta de Negócio:** Como as taxas de atraso se comportam ao longo dos meses? Existe sazonalidade ou tendência de alta/queda?

**Métricas Calculadas:**

 Total de pedidos (`total_pedidos`), quantidade de atrasos (`pedidos_atrasados`) e taxa de atraso percentual (`perc_atrasos`) por mês.

In [0]:
%sql

SELECT
    DATE_FORMAT( order_date, 'MM-yyyy') AS mes,
    COUNT(*) AS total_pedidos,
    SUM(CASE
        WHEN late_delivery = 'Yes' THEN 1
        ELSE 0
    END) AS pedidos_atrasados,
    ROUND( 
        SUM(CASE
            WHEN late_delivery = 'Yes' THEN 1
            ELSE 0
        END)* 100.0 / COUNT (*), 
        2
    )AS perc_atrasos
FROM workspace.default.entregas_tratadas
GROUP BY DATE_FORMAT( order_date, 'MM-yyyy')
ORDER BY mes;

**Insight / Resultado:**

Ao longo de 2026, o percentual mensal de pedidos atrasados permaneceu relativamente estável, variando entre 55,48% e 57,74%. Março apresentou o maior percentual de atrasos, enquanto abril registrou o menor. Como mais da metade dos pedidos ficou atrasada em todos os meses, os resultados indicam que o problema é recorrente ao longo do ano, sem uma tendência clara de aumento ou redução.